# 第五课 数字冰壶比赛中的战术库使用说明

> 鸣谢：感谢东华大学荣智海教授与其指导的赵维同学、赵毅同学共同开源战术库，为参赛同学提供了更好的学习资料与参赛支持。

在前面的课程中，我们已经了解了数字冰壶比赛的通信协议、运动模型与对战策略。本课将结合平台最新接入的战术库代码，系统讲解战术库中 16 个策略函数的分类方式、核心思路以及在实战中的调用方式。

这一课的目标是能清晰解答三个问题：战术库里都有什么、每类战术在什么时候用、代码里应该如何调用。

## 5.1 战术库的总体结构

经过对全部 16 个策略函数的梳理，可以将它们归纳为五种战术类型。这五类基本覆盖了冰壶比赛中从进攻到防守的常见场景。

|类型编号|类型名称|包含策略|核心目标|
|:---:|:---:|:---|:---|
|1|清除类|clear、take_out|直接把对方壶打飞|
|2|防守类|defense、defense_push_in、disarm_defend、freeze|保护己方壶或阻碍对方进攻|
|3|双飞类|double_hit_init、double_hit_gote、double_hit_last|一杆同时打飞两个对方壶|
|4|传击类|push_in、push_in_14、double_push_in、double_push_in_center|借力打力，用己方壶作中间壶|
|5|策略类|middle_in_center、occupy、hit_roll|抢分、控场、制造优势局面|

从代码分层上看，`strategy_library.py` 负责输出某种具体打法，而 `base_library.py` 负责提供路径判断、距离计算和经验查表等底层能力。

`tacticslib/data` 目录是战术库的参数数据目录，目前主要包含 `data.json`。这个文件中保存的不是比赛流程代码，而是大量供基础库直接读取的经验数据和查表数据，例如 `feature`、`v`、`feature_roll`、`v_roll`、`feature_roll_pro`、`h_roll_pro`、`vy`、`vh` 等数组。

这些数据的作用主要有三类：

- 为 `feature_v`、`feature_roll_v`、`feature_roll_v_pro` 这类函数提供查表依据，用来把“目标位置、夹角、碰撞关系”等几何特征转换成可执行的速度或偏移量；
- 为 `hit_roll`、`push_in`、`double_push_in_center` 等复杂战术提供经验参数，避免每次都从零推导完整碰撞结果；
- 为 `np.polyfit(vy, vh, 4)` 这样的拟合过程提供样本点，使基础库能够把纵向位置进一步映射成较平滑的出手速度修正。

可以把这个目录理解成战术库背后的“经验参数仓库”：上层战术函数决定要打什么球，基础库决定该怎么计算，而 `data` 目录则负责提供这些计算所依赖的历史样本、查表数组和拟合参数。

## 5.2 清除类战术

代表策略：`clear`（炸球）、`take_out`（打定）

清除类策略的核心目标是把对方已经形成威胁的壶直接清掉。它们在战术上的区别主要在于：

- `clear` 更适合处理多个壶聚集的局面，尝试一次击打形成连锁碰撞；
- `take_out` 更适合点名清理离圆心最近的对方壶；
- 两者都非常依赖路径是否畅通，因此会频繁调用 `Roadjudge` 进行几何判定。

通俗理解：`clear` 像是扫荡模式，发现一串壶挤在一起就一杆打散；`take_out` 像是定点清除，优先把最危险的那一颗打掉。

### 5.2.1 `clear`（炸球）代码详解

`clear` 的核心目标是处理多个壶聚集的局面。它的思路不是单纯瞄准某一颗壶，而是先找出横向位置非常接近的一组壶，再通过一次击打制造连锁碰撞。

输入参数：

- `state_list`：场上所有壶的坐标列表，格式为 `[[x1, y1], [x2, y2], ...]`。
- `is_init`：先后手标识，这个策略中基本不直接使用。
- `shot_num`：当前投球序号，这个策略中基本不直接使用。
- 返回值：`[v0, h0, 0]` 或 `None`。

代码的主要处理流程如下：

1. 先按纵坐标从大到小排序，并过滤掉坐标为 `(0, 0)` 的出界壶。
2. 扫描场上是否存在“密集区”：如果有 3 个或以上的壶，横坐标差值小于 `2 * Stone_R`，就认为它们挤在一起。
3. 对这组壶重新排序后，取其中靠前的两个关键目标，计算标准方向向量。
4. 通过 `multiple = 2 * Stone_R / 距离` 计算偏移倍率，把瞄准点放到目标壶后方 `2 * Stone_R` 的位置。
5. 最后返回固定速度 `6`，以及相对于大本营中心 `x = 2.375` 的横向偏移量。

通俗理解就是：先找到“一串排得很近的壶”，再让打出去的壶撞上第二颗，借它去带飞第一颗，从而实现“一串二”甚至“一串多”的效果。

### 5.2.2 `take_out`（打定）代码详解

`take_out` 更像是一个定点清除策略，目标通常是离大本营圆心最近、威胁最大的对方壶。

代码的主要思路如下：

1. 先根据 `is_init` 将 `state_list` 分离成 `state_enemy` 和 `state_me` 两组。
2. 通过基础库中的 `NearestTarget`，找出离圆心 `(2.375, 4.88)` 最近的对方壶。
3. 使用 `Roadjudge` 判断从发球点 `(x, 10.61)` 到该目标壶的路径是否畅通。
4. 如果畅通，就直接以固定速度 `6` 击打目标壶，偏移量为目标壶的 `x` 坐标减去圆心 `x` 坐标。
5. 如果路径被挡住，则在目标壶左右各约 `0.145` 的范围内做微调，寻找一条可以绕开障碍的直线路径。

它的算法重点不在复杂碰撞，而在于“先找最危险的壶，再想办法把这一击打进去”。因此 `take_out` 是一种目标非常明确、执行也相对直接的清除策略。

## 5.3 防守类战术

代表策略：`defense`、`defense_push_in`、`disarm_defend`、`freeze`

防守类策略并不是消极等待，而是通过布置保护壶、阻挡传击路线、贴住对方壶等手段降低对手得分概率。它们的典型用途如下：

- `freeze`：贴住目标壶，限制后续击打空间；
- `defense`：当己方中心壶领先但暴露时，在前方补一个保护壶；
- `defense_push_in`：预判对方可能的传击线路，在连线中点上放壶阻断；
- `disarm_defend`：在对方关键壶两侧落壶，形成夹击式限制。

这一类战术的共同特点是：通常不直接追求本杆得分，而是通过改变场面结构，让对手的下一步更难走。

### 5.3.1 `freeze`（粘球）代码详解

`freeze` 的目标不是把对方壶打飞，而是把己方壶精准地贴到对方壶旁边，限制对方下一步处理该壶的空间。由于两壶距离很近，对方如果想直接打飞它，往往也会把自己的壶一并带偏。

代码大致按下面的逻辑展开：

1. 与 `take_out` 一样，先分离敌我壶。
2. 对对方壶按距离圆心远近排序，并过滤掉已经出界的壶。
3. 遍历对方壶时，先计算该壶与圆心连线和水平方向的夹角 `theta`，再结合其距离 `d` 判断这个目标是否值得粘。
4. 如果目标位置、角度和距离都合适，再用 `Roadjudge` 检查从发球点到目标壶、以及目标壶附近的路径是否畅通。
5. 条件满足时，通过 `y2v` 根据目标壶的纵坐标计算合适速度，使己方壶刚好贴到目标壶旁边。

因此 `freeze` 的关键不在于速度大，而在于选一个“值得粘”的目标，然后用合适的速度把己方壶停在非常贴近的位置。

### 5.3.2 `defense`（防守）代码详解

`defense` 的核心目标是保护己方已经占据优势的位置，尤其是当己方中心壶比对方更靠近圆心时。

它的主要判断逻辑是：

1. 找到己方距离圆心最近的壶 `out_s[0]`，以及对方距离圆心最近的壶 `enemy_s[0]`。
2. 如果己方中心壶更接近圆心，且距离圆心已经足够近，就说明己方当前有得分优势。
3. 这时再判断从发球点到己方中心壶的路径是否畅通。
4. 如果畅通，说明这颗得分壶前方没有保护，就需要补一个保护壶；普通壶次下通常使用较慢速度 `2.7`。
5. 如果路径不畅通，还会继续判断挡路的壶是否是对方壶；若是，则直接粘住那颗挡路壶，把被动局面转化成主动防守。

`defense` 的本质就是一句话：己方已经占到便宜时，不要急着继续进攻，而是先保护好已有优势。

### 5.3.3 `defense_push_in`（防传击）代码详解

`defense_push_in` 面向的是一种更具体的威胁：对方可能通过“传击”线路，把前方壶撞向己方中心壶或圆心区域。

它的核心处理流程如下：

1. 遍历对方壶，计算该壶与己方中心壶之间的连线方向。
2. 如果这条连线的夹角在约 `40°` 到 `90°` 之间，且对方壶位于上方，就说明对方很可能沿着这条斜线发起传击。
3. 若己方中心壶非常接近圆心，就在敌方壶与己方中心壶连线的中点上放保护壶；否则就在敌方壶与圆心连线的中点上放壶。
4. 最后用 `Roadjudge` 和 `LocalJudge` 同时检查路径可达性和落点可行性。

这个策略的要点在于“预判线路”而不是“事后补救”。如果判断出对方存在明显传击路径，就提前在中点上布一个壶，把这条路直接堵死。

### 5.3.4 `disarm_defend`（分壶防守）代码详解

`disarm_defend` 可以理解为一种“夹击式防守”。它主要寻找位于中心附近、且上方路径相对畅通的对方壶，然后尝试在其左右两侧的空位落壶，限制对方后续操作空间。

它的基本思路是：

1. 只处理处于大本营核心范围附近的对方壶。
2. 检查这些壶上方是否已经被己方壶阻挡，如果完全无路可走就不必再处理。
3. 如果目标壶位于中心左侧，则优先尝试在其右侧布壶；若位于中心右侧，则优先在其左侧布壶。
4. 一旦找到空位，就用较慢的速度完成落壶，形成左右夹击效果。

相比直接清壶，这类防守更像是在给对方“套笼子”：不一定立刻得分，但会让对方的下一步明显更难处理。

## 5.4 双飞类战术

代表策略：`double_hit_init`、`double_hit_gote`、`double_hit_last`

双飞类的核心逻辑是一次击打同时处理两个对方壶。它的难点不在于单纯地打中一颗壶，而在于让第一碰撞球在受力后继续准确撞向第二碰撞球。

代码中这类策略的主要流程一般是：

1. 找到离圆心最近的对方壶作为第一碰撞球；
2. 在其余对方壶中寻找能构成双飞几何关系的第二碰撞球；
3. 计算碰撞角度、辅瞄点和击打点；
4. 用经验阈值过滤掉成功率低的角度组合；
5. 检查发球点到击打点、击打点到第一碰撞球的路径是否畅通。

`double_hit_last` 是最后时刻的激进版本，约束更宽松，适合末投搏杀。

### 5.4.1 `double_hit_init` / `double_hit_gote` 代码详解

双飞类策略的核心是：一次击打，同时处理两个对方壶。`double_hit_init` 和 `double_hit_gote` 的主体结构基本一致，区别主要在于适用场景的判断条件略有不同。

整体思路可以分为五步：

1. 先分离敌我壶，并找到距离圆心最近的对方壶，作为“第一碰撞球”。
2. 在其他对方壶中寻找可以作为“第二碰撞球”的目标，要求它通常位于第一碰撞球的上方，或者也足够靠近中心区域。
3. 根据两颗目标壶之间的距离，选择不同的几何计算方式，求出辅助瞄准点。
4. 通过若干经验阈值过滤掉成功率较低的角度和距离组合。
5. 最后结合两壶的相对方位，求出击打点，并用 `Roadjudge` 检查从发球点到击打点、从击打点到第一碰撞球的路径是否都畅通。

可以把这个策略理解成冰壶里的“一石二鸟”：真正难的不是打中第一颗壶，而是要让第一颗壶在被撞之后还能按预期继续飞向第二颗壶。

### 5.4.2 `double_hit_last`（最后双飞）代码详解

`double_hit_last` 是双飞策略的末投版本。它和 `double_hit_init` 的总体结构非常接近，但有两个显著特征：

- 默认服务于最后一投附近的关键回合；
- 在筛选第二碰撞球时条件更宽松，允许采取更激进的高风险打法。

因此它更像是一种“最后一搏”策略：如果常规保守打法已经不足以扭转局面，就放宽约束，争取通过一记双飞直接改变得分格局。

## 5.5 传击类战术

代表策略：`push_in`、`push_in_14`、`double_push_in`、`double_push_in_center`

传击类战术的本质是利用己方已有壶作为中间壶，把力量一层层传递给前方目标。与直接打定相比，这种打法可以绕开部分障碍，也可以让原本无法直达的位置变得可达。

其中：

- `push_in`：单级传击，目标通常是离圆心最近的对方壶；
- `push_in_14`：末投版本，允许更激进的传击，找不到目标时甚至会把圆心本身当作目标；
- `double_push_in`：两级接力，通过两个己方中间壶形成更远距离的传击；
- `double_push_in_center`：目标不是对方壶，而是把己方壶准确送进圆心。

这类策略对角度、上下位置关系和路径畅通性的要求都很高，因此代码中会大量使用距离公式、向量关系和 `feature_v`、`y2_v` 一类经验速度映射函数。

### 5.5.1 `push_in` 与 `push_in_14` 代码详解

传击类策略的共同特点是：并不是直接去打目标壶，而是先击打己方中间壶，再把力量传递出去。

`push_in` 的基本逻辑如下：

1. 先分离敌我壶，并找出距离圆心最近的对方壶作为主要目标。
2. 遍历己方壶，检查哪些壶可以充当中间壶。中间壶必须位于目标壶下方，且两者连线的夹角要足够大，否则传递效果太差。
3. 一旦找到合适的中间壶，就计算辅助瞄准点，也就是打出壶应该撞击中间壶的具体位置。
4. 再用 `Roadjudge` 检查从发球点到辅助瞄准点、从中间壶到目标壶这两段路径是否都可行。
5. 如果存在多个可行方案，则优先选择距离圆心最近、威胁最大的目标壶。

`push_in_14` 则是末投版本。它与 `push_in` 的区别在于条件更宽松，允许更激进地尝试传击；如果实在找不到合适的对方目标壶，甚至会直接把圆心 `(2.375, 4.88)` 当成目标，尝试把己方壶传进得分区。

从战术视角看，`push_in` 强调的是“借力打力”，而 `push_in_14` 则是在最后阶段把这种借力打法用到更激进的程度。

### 5.5.2 `double_push_in` 与 `double_push_in_center` 代码详解

`double_push_in` 可以看作是传击类策略的双级接力版本。它的目标不是只靠一个中间壶完成传递，而是让力量经过两次接力，最终作用到目标壶上。

代码思路可以概括为：

1. 先找到距圆心最近的对方壶作为最终目标。
2. 找到第一个位于目标下方、角度合适的己方壶，作为中间壶 1。
3. 再找到第二个位于中间壶 1 下方、同样具备可行几何关系的壶，作为中间壶 2。
4. 依次计算两次接力所需的瞄准点与路径，并验证路径是否畅通。

这相当于把“打出壶 -> 中间壶 1 -> 中间壶 2 -> 目标壶”串成了一条更长的传递链路。

`double_push_in_center` 的区别在于，最终目标不再是某一颗对方壶，而是圆心 `(2.375, 4.88)`。它的目的不是清壶，而是把己方壶通过两次传递准确送进大本营中心得分。

因此在实现上，`double_push_in_center` 不再简单使用固定速度 `6`，而是结合 `y2_v`、`delta_y2v_big`、`delta_y2v_small` 等函数，对两次传递带来的位移进行速度补偿，从而得到更细致的速度控制。

## 5.6 策略类战术

代表策略：`middle_in_center`、`occupy`、`hit_roll`

策略类战术不一定第一时间清壶，而是通过控场、抢位和落点设计制造长期优势。它们在项目中非常重要，因为这类函数经常出现在前中盘。

- `middle_in_center`：选择条件合适的己方壶，直接把它打进圆心附近；
- `occupy`：判断左、中、右三个关键区域和中心区的局面，决定占中、占边、贴防守或尝试垂直双飞；
- `hit_roll`：在击飞对方壶后，让己方壶继续滑向预设好位置，是兼顾清壶与落位的高级战术。

其中 `occupy` 是策略树最复杂的函数之一，`hit_roll` 则是经验查表最重的函数之一。前者偏局面判断，后者偏轨迹控制。

### 5.6.1 `middle_in_center` 与 `occupy` 代码详解

策略类战术和前面的清除、双飞、传击不同，它们更多是在主动塑造局面。

`middle_in_center` 的核心目标很直接：在中心区域为空或合适时，把己方壶直接送进圆心附近得分。它的大致流程是：

1. 先检查中心附近是否已经有壶，如果已有壶，就不再执行这个策略。
2. 在己方壶中寻找候选目标，这些候选壶需要满足三个条件：位于大本营上方、横向位置在中心附近、到发球点和圆心的路径都畅通。
3. 在候选壶中选择距离圆心最近的一个。
4. 再结合 `d_v`、`v_d` 和多项式修正，计算最终应该使用的速度与偏移量。

相比之下，`occupy` 是策略库中更复杂的控场函数。它并不是只盯着圆心，而是综合判断左、中、右三个区域，以及中心区域的壶分布情况，然后决定下一步应该占中、占左、占右，还是尝试通过 `double_hit_vertical` 处理特殊局面。

换句话说，`middle_in_center` 更偏向直接得分，而 `occupy` 更偏向布局和抢位。

### 5.6.2 `hit_roll`（打甩）代码详解

`hit_roll` 是策略类中技术含量很高的一种打法。它不仅要求把对方壶击走，还要求己方壶在碰撞后继续滑向一个理想位置。

这类策略的难点在于：不同横向位置、不同纵向位置的目标壶，对应的碰撞几何关系和速度控制都不一样。

因此代码中把目标壶按距离中心的远近划分成多个区域，例如：

- `middle`：`abs(x - 2.375) < 1`
- `low`：`1 <= abs(x - 2.375) < 1.5`
- `max`：与 `low` 接近，但根据纵向位置采用不同处理
- `extension`：`1.5 <= abs(x - 2.375) <= 2`

不同区域会使用不同的经验查表函数：

- `feature_roll_v_pro` 负责中间区域的横向偏移量估计；
- `feature_roll_v_max` 负责更极端角度下的偏移；
- `feature_roll_v` 和 `feature_roll_v_extension` 则负责更复杂场景下的速度估计。

最终，`hit_roll` 会在可行目标中优先选择更靠近圆心的壶，并根据区域不同返回固定速度或查表速度。这也是为什么它既像清壶，又像落位控制，是一个兼顾进攻与布局的高级战术。

## 5.7 如何在当前项目中调用战术库

首先点击页面左上角Jupyter菜单中的[Run]菜单项，点击该该菜单项的[Start Curling Server]子菜单项，启动数字冰壶比赛服务器，点击数字冰壶比赛服务器界面中的【四局制】按钮进入初赛模式。

下方这段范例代码其实就是当前项目中 `tacticslib/main.py` 的源码。课程把它直接放在这里，是为了结合真实项目中的主程序，说明战术库是如何在 `TacticsRobot` 中被接入、分发并最终参与实际对局的。

在这段范例代码中，`TacticsRobot` 继承自 `AIRobot`，并通过重写父类的 `get_bestshot()` 方法，在合适的壶次调用战术库中的不同函数来修改投掷策略。整体调用链如下：

1. `TacticsRobot.recv_forever()` 持续接收服务器消息；
2. 收到 `GO` 消息后调用 `TacticsRobot.get_bestshot()`；
3. `get_bestshot()` 根据先手/后手进入 `Strategy_init()` 或 `Strategy_gote()`；
4. 两个函数根据当前是我方第几壶，决定调用 `occupy`、`hit_roll`、`defense`、`push_in` 等具体战术；
5. 战术函数返回 `[v0, h0, w0]` 后，再封装成 `BESTSHOT v0 h0 w0` 发给服务器。

需要注意的是，这段范例代码并不是为了实现主动搜索最优策略，而是为了按照预先设计好的剧本完成一场表演赛，从而更直观地展示战术库中的各类函数是如何被调用、分发和使用的。

之所以要这样设计，是因为<b>战术库中的每一种战术都只能在适合它的特定场景下才有可能被成功调用</b>。如果只是随意进行一场普通对局，很多战术可能根本没有机会触发。因此本课在最后专门安排了这样一场表演赛，用来为这些战术创造合适的局面，方便观察它们的实际调用过程。

<b>注意要根据数字冰壶服务器界面中提供的信息修改变量key的赋值</b>。运行下方范例代码启动JupyterAI选手，在数字冰壶服务器界面中可以看到＜Player1 已连接＞。

参照4.3.2中给出的步骤，在控制台中运行tacticslib/main.py脚本启动TacticsAI选手，在数字冰壶服务器界面中可以看到＜Player2 已连接＞。这样就可以实现tacticslib/main.py脚本与自身的对战。

在数字冰壶服务器界面中点击【准备】按钮，再点击【开始对局】按钮，即可在JupyterAI和TacticsAI的对战中检验前面接入的战术库是否能够按照预期工作。你可以重点观察不同壶次下AI是否会进入 `occupy`、`hit_roll`、`push_in`、`defense` 等对应分支，并结合比赛结果判断当前策略树的设置是否合理。

需要说明的是，受限于各类战术往往只会在特定局面下才有机会被触发，这场表演赛并没有覆盖并调用到全部战术。对于本次演示中尚未触发的战术，它们在真实对局中的具体效果，还有待大家在后续实战中进一步观察、测试和探索。

## 5.8 小结

可以把调用当前项目中的战术库的过程理解为制定一个“规则策略引擎”：

- 上层按比赛阶段、先后手与壶次决定使用哪种战术；
- 中层的战术函数负责把抽象意图变成可执行的击打方案；
- 下层的基础库负责距离、路径、角度和速度的具体计算。

当你需要修改比赛策略时，优先调整 `tacticslib/main.py` 中的战术分发；当你需要提升某一种打法的成功率时，再深入修改 `strategy_library.py` 或 `base_library.py` 中对应的几何与经验参数。

在后续使用和改进战术库时，可以优先思考两个问题：当前局面适合触发哪一类战术，以及现有分发和参数设置是否真的符合你的比赛目标。

进一步看，战术库的意义还不只是在规则对局中直接调用这些策略，它也可以为后续的强化学习算法提供一组具有明确战术含义的候选动作。这样做能够把原本连续、分散、难以约束的随机参数搜索，转化为先在这些候选战术动作之间进行选择、再做进一步优化，从而在一定程度上缓解训练过程中难以收敛的问题。